In [ ]:

import os
import gymnasium as gym
import ale_py
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import time
import pickle

from gymnasium.wrappers import GrayScaleObservation, ResizeObservation, FrameStack

# Settings
LOAD_CHECKPOINT = "mario_checkpoint.pth"
SAVE_CHECKPOINT = "mario_checkpoint_resume.pth"
REPLAY_DUMP = "replay_buffer_resume.pkl"
LOG_FILE = "training_log_resume.txt"

DESIRED_START_EP = 1
END_EPISODE = 400

ALLOWED_ACTIONS = [0, 2, 3, 4, 7]
GAMMA = 0.99
BATCH_SIZE = 32
TARGET_UPDATE = 2000
LR = 5e-5
REPLAY_SIZE = 200_000
WARMUP_MIN_EXPERIENCES = 50_000
EVAL_INTERVAL = 10
EVAL_EPISODES = 3
SAVE_EVERY_EP = 10
MAX_GRAD_NORM = 10.0

EPS_START = 1.0
EPS_FINAL = 0.05
EPS_DECAY_STEPS = 250_000

# Environment
env = gym.make("ALE/MarioBros-v5", render_mode="rgb_array")
env = GrayScaleObservation(env)
env = ResizeObservation(env, (84, 84))
env = FrameStack(env, 4)

obs_shape = env.observation_space.shape
n_actions = len(ALLOWED_ACTIONS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DQN Network
class DQN(nn.Module):
    def __init__(self, shape, n_act):
        super().__init__()
        c = shape[0]
        self.net = nn.Sequential(
            nn.Conv2d(c, 32, 8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, 512), nn.ReLU(),
            nn.Linear(512, n_act)
        )
    def forward(self, x):
        x = x / 255.0
        return self.net(x)

# Replay Buffer
class ReplayBuffer:
    def __init__(self, size=200_000):
        self.buffer = deque(maxlen=size)

    def push(self, s, a, r, ns, d):
        self.buffer.append((np.array(s, copy=False),
                            int(a),
                            float(r),
                            np.array(ns, copy=False),
                            bool(d)))

    def sample(self, n):
        batch = random.sample(self.buffer, n)
        return tuple(np.array(x) for x in zip(*batch))

    def __len__(self):
        return len(self.buffer)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(list(self.buffer), f)

    def load(self, path):
        with open(path, "rb") as f:
            data = pickle.load(f)
            self.buffer = deque(data, maxlen=self.buffer.maxlen)

# Helpers
def epsilon_by_step(step):
    frac = min(1.0, step / EPS_DECAY_STEPS)
    return EPS_START + frac * (EPS_FINAL - EPS_START)

def select_action(state, eps, net):
    if random.random() < eps:
        return random.randrange(n_actions)
    with torch.no_grad():
        s = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        return net(s).argmax(1).item()

# Initialize
policy_net = DQN(obs_shape, n_actions).to(device)
target_net = DQN(obs_shape, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
buffer = ReplayBuffer(REPLAY_SIZE)

global_step = 0
start_episode = DESIRED_START_EP

# Load checkpoint
if os.path.exists(LOAD_CHECKPOINT):
    ckpt = torch.load(LOAD_CHECKPOINT, map_location=device)
    policy_net.load_state_dict(ckpt["policy"])
    target_net.load_state_dict(ckpt["target"])
    try:
        optimizer.load_state_dict(ckpt["optimizer"])
    except:
        pass
    global_step = ckpt.get("steps", global_step)
    if ckpt.get("episode"):
        start_episode = ckpt["episode"] + 1

# Load replay buffer
if os.path.exists(REPLAY_DUMP):
    try:
        buffer.load(REPLAY_DUMP)
    except:
        pass

# Logging
log_f = open(LOG_FILE, "a")
def log(msg):
    log_f.write(msg + "\n")
    log_f.flush()

# Warmup buffer
def warmup(min_size):
    if len(buffer) >= min_size:
        return
    while len(buffer) < min_size:
        obs, _ = env.reset()
        s = np.array(obs)
        done = False
        while not done and len(buffer) < min_size:
            a = random.randrange(n_actions)
            action = ALLOWED_ACTIONS[a]
            next_obs, r, t, tr, _ = env.step(action)
            ns = np.array(next_obs)
            done = t or tr
            shaped = r * 0.01 + 0.1
            buffer.push(s, a, shaped, ns, done)
            s = ns

warmup(WARMUP_MIN_EXPERIENCES)

# Evaluation (numeric only)
def evaluate(net):
    results = []
    for _ in range(EVAL_EPISODES):
        obs, _ = env.reset()
        s = np.array(obs)
        done = False
        steps = 0
        while not done:
            a_idx = select_action(s, 0.01, net)
            nxt, r, t, tr, _ = env.step(a_idx)
            s = np.array(nxt)
            done = t or tr
            steps += 1
        results.append(steps)
    return float(np.mean(results)), results

# Training Loop
best_eval = -1

try:
    for episode in range(start_episode, END_EPISODE + 1):

        obs, _ = env.reset()
        state = np.array(obs)
        done = False
        steps = 0
        raw_total = 0.0
        shaped_total = 0.0

        epsilon = epsilon_by_step(global_step)

        while not done:
            steps += 1
            global_step += 1

            a_idx = select_action(state, epsilon, policy_net)
            action = ALLOWED_ACTIONS[a_idx]

            next_obs, reward, t, tr, _ = env.step(action)
            next_state = np.array(next_obs)
            done = t or tr

            raw_total += reward
            shaped = reward * 0.01 + 0.1
            shaped_total += shaped

            buffer.push(state, a_idx, shaped, next_state, done)
            state = next_state

            # training
            if len(buffer) > BATCH_SIZE:
                s, a, r, ns, d = buffer.sample(BATCH_SIZE)
                s = torch.tensor(s, device=device, dtype=torch.float32)
                a = torch.tensor(a, device=device).unsqueeze(1)
                r = torch.tensor(r, device=device)
                ns = torch.tensor(ns, device=device, dtype=torch.float32)
                d = torch.tensor(d, device=device, dtype=torch.float32)

                q = policy_net(s).gather(1, a.long()).squeeze(1)

                with torch.no_grad():
                    na = policy_net(ns).argmax(1)
                    nq = target_net(ns).gather(1, na.unsqueeze(1)).squeeze(1)

                target = r + GAMMA * nq * (1 - d)
                loss = nn.functional.smooth_l1_loss(q, target)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_net.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            if global_step % TARGET_UPDATE == 0:
                target_net.load_state_dict(policy_net.state_dict())

        log(f"Episode {episode}/{END_EPISODE} | Steps {steps} | Raw {raw_total:.1f} | Shaped {shaped_total:.2f} | Eps {epsilon:.4f}")

        # Save checkpoints
        if episode % SAVE_EVERY_EP == 0:
            torch.save({
                "policy": policy_net.state_dict(),
                "target": target_net.state_dict(),
                "optimizer": optimizer.state_dict(),
                "steps": global_step,
                "episode": episode
            }, SAVE_CHECKPOINT)
            buffer.save(REPLAY_DUMP)

        # Eval
        if episode % EVAL_INTERVAL == 0:
            mean_steps, arr = evaluate(policy_net)
            log(f"Eval ep {episode}: mean {mean_steps:.1f} | {arr}")
            if mean_steps > best_eval:
                best_eval = mean_steps
                torch.save({
                    "policy": policy_net.state_dict(),
                    "target": target_net.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "steps": global_step,
                    "episode": episode
                }, SAVE_CHECKPOINT.replace(".pth", "_best.pth"))

        if episode%50==0: 
            time.sleep(180)

finally:
    try:
        torch.save({
            "policy": policy_net.state_dict(),
            "target": target_net.state_dict(),
            "optimizer": optimizer.state_dict(),
            "steps": global_step,
            "episode": episode
        }, SAVE_CHECKPOINT)
    except:
        pass
    try:
        buffer.save(REPLAY_DUMP)
    except:
        pass
    log_f.close()
    env.close()


In [ ]:
import os
import gymnasium as gym
import ale_py
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import pickle

from gymnasium.wrappers import GrayScaleObservation, ResizeObservation, FrameStack

# --- Settings ---
LOAD_CHECKPOINT = "mario_checkpoint.pth"
SAVE_CHECKPOINT = "mario_checkpoint_resume.pth"
LOG_FILE = "training_log_resume.txt"

# Buffer saving is risky (high RAM usage). 
# Only enable if you have >32GB RAM and fast SSD.
ENABLE_BUFFER_SAVE = False 
REPLAY_DUMP = "replay_buffer_resume.pkl"

DESIRED_START_EP = 1
END_EPISODE = 500

ALLOWED_ACTIONS = [0, 2, 3, 4, 7] # NOOP, UP, RIGHT, LEFT, RIGHTFIRE
GAMMA = 0.99
BATCH_SIZE = 32
TARGET_UPDATE = 2000
LR = 5e-5
REPLAY_SIZE = 200_000
WARMUP_MIN_EXPERIENCES = 20_000 # Reduced slightly for faster startup
EVAL_INTERVAL = 10
EVAL_EPISODES = 3
SAVE_EVERY_EP = 10
MAX_GRAD_NORM = 10.0

EPS_START = 1.0
EPS_FINAL = 0.05
EPS_DECAY_STEPS = 250_000

# --- Environment Setup ---
def make_env():
    env = gym.make("ALE/MarioBros-v5", render_mode="rgb_array")
    env = GrayScaleObservation(env)
    env = ResizeObservation(env, (84, 84))
    env = FrameStack(env, 4)
    return env

env = make_env()
obs_shape = env.observation_space.shape
n_actions = len(ALLOWED_ACTIONS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# --- DQN Network (Nature CNN) ---
class DQN(nn.Module):
    def __init__(self, shape, n_act):
        super().__init__()
        c = shape[0]
        self.net = nn.Sequential(
            nn.Conv2d(c, 32, 8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3136, 512), nn.ReLU(),
            nn.Linear(512, n_act)
        )

    def forward(self, x):
        # Normalize pixel values [0, 255] -> [0, 1]
        x = x / 255.0
        return self.net(x)

# --- Replay Buffer ---
class ReplayBuffer:
    def __init__(self, size=200_000):
        self.buffer = deque(maxlen=size)

    def push(self, s, a, r, ns, d):
        # Store as numpy arrays to save space compared to tensors
        self.buffer.append((np.array(s, copy=False),
                            int(a),
                            float(r),
                            np.array(ns, copy=False),
                            bool(d)))

    def sample(self, n):
        batch = random.sample(self.buffer, n)
        return tuple(np.array(x) for x in zip(*batch))

    def __len__(self):
        return len(self.buffer)

    def save(self, path):
        if not ENABLE_BUFFER_SAVE: return
        print("Saving replay buffer (this may take a while)...")
        try:
            with open(path, "wb") as f:
                pickle.dump(list(self.buffer), f)
        except Exception as e:
            print(f"Buffer save failed: {e}")

    def load(self, path):
        if not ENABLE_BUFFER_SAVE: return
        if not os.path.exists(path): return
        print("Loading replay buffer...")
        try:
            with open(path, "rb") as f:
                data = pickle.load(f)
                self.buffer = deque(data, maxlen=self.buffer.maxlen)
        except Exception as e:
            print(f"Buffer load failed: {e}")

# --- Helpers ---
def epsilon_by_step(step):
    frac = min(1.0, step / EPS_DECAY_STEPS)
    return EPS_START + frac * (EPS_FINAL - EPS_START)

def select_action(state, eps, net):
    if random.random() < eps:
        return random.randrange(n_actions)
    
    with torch.no_grad():
        # state is (4, 84, 84), needs to be (1, 4, 84, 84)
        s = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        return net(s).argmax(1).item()

# --- Initialization ---
policy_net = DQN(obs_shape, n_actions).to(device)
target_net = DQN(obs_shape, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=LR)
buffer = ReplayBuffer(REPLAY_SIZE)

global_step = 0
start_episode = DESIRED_START_EP

# Load Checkpoint
if os.path.exists(LOAD_CHECKPOINT):
    print(f"Loading checkpoint: {LOAD_CHECKPOINT}")
    ckpt = torch.load(LOAD_CHECKPOINT, map_location=device)
    policy_net.load_state_dict(ckpt["policy"])
    target_net.load_state_dict(ckpt["target"])
    try:
        optimizer.load_state_dict(ckpt["optimizer"])
    except:
        print("Optimizer state mismatch, starting fresh optimizer.")
    global_step = ckpt.get("steps", global_step)
    if ckpt.get("episode"):
        start_episode = ckpt["episode"] + 1

# Load Buffer (Optional)
if os.path.exists(REPLAY_DUMP) and ENABLE_BUFFER_SAVE:
    buffer.load(REPLAY_DUMP)

# Logging
log_f = open(LOG_FILE, "a")
def log(msg):
    print(msg)
    log_f.write(msg + "\n")
    log_f.flush()

# --- Warmup ---
def warmup(min_size):
    if len(buffer) >= min_size:
        return
    print(f"Warming up buffer ({len(buffer)}/{min_size})...")
    while len(buffer) < min_size:
        obs, _ = env.reset()
        s = np.array(obs)
        done = False
        while not done and len(buffer) < min_size:
            a = random.randrange(n_actions)
            action = ALLOWED_ACTIONS[a]
            next_obs, r, t, tr, _ = env.step(action)
            ns = np.array(next_obs)
            done = t or tr
            
            # Simple reward clipping
            shaped = np.clip(r, -1, 1)
            buffer.push(s, a, shaped, ns, done)
            s = ns

warmup(WARMUP_MIN_EXPERIENCES)

# --- Evaluation Function (FIXED) ---
def evaluate(net):
    results = []
    for _ in range(EVAL_EPISODES):
        obs, _ = env.reset()
        s = np.array(obs)
        done = False
        steps = 0
        while not done:
            # Always deterministic (0.01 eps) during eval
            a_idx = select_action(s, 0.01, net)
            
            # [FIX]: Map index to ALLOWED_ACTIONS
            action = ALLOWED_ACTIONS[a_idx]
            
            nxt, r, t, tr, _ = env.step(action)
            s = np.array(nxt)
            done = t or tr
            steps += 1
        results.append(steps)
    return float(np.mean(results)), results

# --- Main Training Loop ---
best_eval = -1
print("Starting training...")

try:
    for episode in range(start_episode, END_EPISODE + 1):
        obs, _ = env.reset()
        state = np.array(obs)
        done = False
        steps = 0
        raw_total = 0.0
        shaped_total = 0.0

        epsilon = epsilon_by_step(global_step)

        while not done:
            steps += 1
            global_step += 1

            # 1. Select Action
            a_idx = select_action(state, epsilon, policy_net)
            action = ALLOWED_ACTIONS[a_idx]

            # 2. Step Environment
            next_obs, reward, t, tr, _ = env.step(action)
            next_state = np.array(next_obs)
            done = t or tr

            raw_total += reward
            
            # Reward shaping: Clip or Scale
            # reward * 0.01 + 0.1 is okay for survival, 
            # but Mario usually needs specific goal rewards.
            # Using simple clipping for stability here.
            shaped = np.clip(reward, -1, 1) 
            shaped_total += shaped

            # 3. Store
            buffer.push(state, a_idx, shaped, next_state, done)
            state = next_state

            # 4. Train
            if len(buffer) > BATCH_SIZE:
                s, a, r, ns, d = buffer.sample(BATCH_SIZE)
                
                s = torch.tensor(s, device=device, dtype=torch.float32)
                a = torch.tensor(a, device=device).unsqueeze(1)
                r = torch.tensor(r, device=device)
                ns = torch.tensor(ns, device=device, dtype=torch.float32)
                d = torch.tensor(d, device=device, dtype=torch.float32)

                # Current Q
                q = policy_net(s).gather(1, a.long()).squeeze(1)

                # Double DQN Target
                with torch.no_grad():
                    na = policy_net(ns).argmax(1)
                    nq = target_net(ns).gather(1, na.unsqueeze(1)).squeeze(1)
                    target = r + GAMMA * nq * (1 - d)

                loss = nn.functional.smooth_l1_loss(q, target)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_net.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            # Update Target Network
            if global_step % TARGET_UPDATE == 0:
                target_net.load_state_dict(policy_net.state_dict())

        # End of Episode Log
        log(f"Ep {episode} | Steps {steps} | Raw {raw_total:.1f} | Eps {epsilon:.3f}")

        # Save Checkpoint
        if episode % SAVE_EVERY_EP == 0:
            torch.save({
                "policy": policy_net.state_dict(),
                "target": target_net.state_dict(),
                "optimizer": optimizer.state_dict(),
                "steps": global_step,
                "episode": episode
            }, SAVE_CHECKPOINT)
            # Buffer save is disabled by flag at top

        # Evaluation
        if episode % EVAL_INTERVAL == 0:
            mean_steps, arr = evaluate(policy_net)
            log(f"  >> EVAL Ep {episode}: Mean Steps {mean_steps:.1f} | Scores: {arr}")
            
            if mean_steps > best_eval:
                best_eval = mean_steps
                torch.save({
                    "policy": policy_net.state_dict(),
                    "target": target_net.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "steps": global_step,
                    "episode": episode
                }, SAVE_CHECKPOINT.replace(".pth", "_best.pth"))
                log(f"  >> New Best Model Saved!")

finally:
    # Save on exit (Ctrl+C)
    print("Saving final state...")
    try:
        torch.save({
            "policy": policy_net.state_dict(),
            "target": target_net.state_dict(),
            "optimizer": optimizer.state_dict(),
            "steps": global_step,
            "episode": episode
        }, SAVE_CHECKPOINT)
    except:
        pass
    
    # Try to save buffer if enabled
    if ENABLE_BUFFER_SAVE:
        buffer.save(REPLAY_DUMP)
        
    log_f.close()
    env.close()
    print("Training finished.")